# Notebook 1: Data Understanding & Preprocessing
    
**BT4222 Project - JDSearch Re-ranking Task**

This notebook handles the initial data parsing, exploratory data analysis, and temporal retention analysis. We parse the raw `.txt` files into structured tables to be used for model training.


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

BASE = Path(r"C:\Users\Acer\OneDrive - National University of Singapore\BT4222\JDsearch")
USER_TXT = BASE / "user_behavior_data.txt"
ARTIFACT_DIR = BASE / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading user_behavior_data.txt...")
user_df = pd.read_csv(USER_TXT, sep="\t", dtype=str, low_memory=False)
print(f"Shape: {user_df.shape}")
display(user_df.head(3))

Loading user_behavior_data.txt...
Shape: (173831, 7)


,query,candidate_wid_list,candidate_label_list,history_qry_list,history_wid_list,history_type_list,history_time_list
0,63995226602574196100244625575117,36098226_17729172_34921664_68317377_693764_360...,1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,-1_-1_-1_68784030_36547097602574193663289_95...,9227588_52788392_4585436_87422901_57364733_210...,ORD_ORD_CART_CART_CART_CART_CART_CART_CART_CAR...,0_0_22967659_82448_432_10315_51_188_51_254173_...
1,1254963427438884,30857980_44577431_1485384_63372823_19754701_29...,1.0_1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,9755613572694219_46916741_-1_-1_-1_-1_-1_4927...,18256630_46206270_65230659_86878921_32743602_5...,ORD_ORD_ORD_CART_CART_CART_CART_ORD_CART_CART_...,0_4813478_1115463_14061983_920_467229_304_1197...
2,24374376067768939206305,42155506_90092899_4336287_74203839_39478459_37...,1.0_1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,-1_-1,8133925_59478082,CLICK_CLICK,0_27_49660


## 1. Parsing Lists & Formatting

In [ ]:
def safe_split(val, sep="_"):
    if pd.isna(val) or str(val).strip() == "": return []
    return str(val).split(sep)

def safe_split_numeric(val, sep="_"):
    return [float(x) for x in safe_split(val, sep)]

user_df["candidate_wid_list_parsed"]   = user_df["candidate_wid_list"].apply(safe_split_numeric)
user_df["candidate_label_list_parsed"] = user_df["candidate_label_list"].apply(safe_split_numeric)
user_df["history_qry_list_parsed"]     = user_df["history_qry_list"].apply(safe_split)
user_df["history_wid_list_parsed"]     = user_df["history_wid_list"].apply(safe_split)
user_df["history_type_list_parsed"]    = user_df["history_type_list"].apply(safe_split)
user_df["history_time_list_parsed"]    = user_df["history_time_list"].apply(safe_split_numeric)

print("List parsing complete.")
display(user_df[["candidate_wid_list_parsed", "history_wid_list_parsed"]].head(3))

List parsing complete.


,candidate_wid_list_parsed,history_wid_list_parsed
0,"[36098226.0, 17729172.0, 34921664.0, 68317377....","[9227588, 52788392, 4585436, 87422901, 5736473..."
1,"[30857980.0, 44577431.0, 1485384.0, 63372823.0...","[18256630, 46206270, 65230659, 86878921, 32743..."
2,"[42155506.0, 90092899.0, 4336287.0, 74203839.0...","[8133925, 59478082]"


## 2. Save Parsed Interactions

In [ ]:
SESSION_COL = "session_id"
user_df_r = user_df.reset_index().rename(columns={"index": SESSION_COL})

# 1. Candidate Interactions
cand_df = user_df_r[[SESSION_COL, "query", "candidate_wid_list_parsed", "candidate_label_list_parsed"]].copy()
cand_df = cand_df.explode(["candidate_wid_list_parsed", "candidate_label_list_parsed"])
cand_df = cand_df.rename(columns={"query": "query_id", "candidate_wid_list_parsed": "item_id", "candidate_label_list_parsed": "raw_action_label"})
cand_df["item_id"] = cand_df["item_id"].astype("Int64")
cand_df["is_recommended"] = 1
cand_df["action_type"] = cand_df["raw_action_label"].map({0.0: "No Interaction", 1.0: "Click", 2.0: "Cart", 3.0: "Purchase"})
cand_df = cand_df[[SESSION_COL, "query_id", "item_id", "is_recommended", "action_type", "raw_action_label"]]

# 2. Historical Interactions
hist_df = user_df_r[[SESSION_COL, "history_qry_list_parsed", "history_wid_list_parsed", "history_type_list_parsed", "history_time_list_parsed"]].copy()
def zip_history(row):
    qs, ws, ts, tms = row["history_qry_list_parsed"], row["history_wid_list_parsed"], row["history_type_list_parsed"], row["history_time_list_parsed"]
    tms_aligned = tms[1:-1] if len(tms) == len(qs) + 1 else tms
    return list(zip(qs, ws, ts, tms_aligned))

hist_df["zipped_history"] = hist_df.apply(zip_history, axis=1)
hist_df = hist_df.explode("zipped_history")
hist_df[["query_id", "item_id", "action_type", "time_to_next"]] = pd.DataFrame(hist_df["zipped_history"].tolist(), index=hist_df.index)
hist_df["item_id"] = pd.to_numeric(hist_df["item_id"], errors="coerce").astype("Int64")
hist_df["query_id"] = hist_df["query_id"].astype(str)
hist_df["is_queryless"] = (hist_df["query_id"] == "-1").astype(int)
hist_df = hist_df[[SESSION_COL, "query_id", "item_id", "is_queryless", "action_type", "time_to_next"]]

cand_df.to_parquet(ARTIFACT_DIR / "candidate_interactions.parquet", index=False)
hist_df.to_parquet(ARTIFACT_DIR / "historical_interactions.parquet", index=False)
print("Saved interaction tables to artifacts.")

C:\Users\Acer\AppData\Local\Temp\ipykernel_6552\281109117.py:22: Pandas4Warning: Constructing a DataFrame from a list of sequences with mismatched lengths is deprecated and will raise in a future version. The shorter sequences are currently padded with NaN; make all sequences the same length before constructing instead.
  hist_df[["query_id", "item_id", "action_type", "time_to_next"]] = pd.DataFrame(hist_df["zipped_history"].tolist(), index=hist_df.index)


Saved interaction tables to artifacts.
